# spotify_client

HTTP client for the Spotify Web API.

Features:
- Automatic Bearer token injection via `token_manager`
- 429 rate-limit backoff (`Retry-After` header)
- 401 token invalidation + one retry
- Deterministic request hash for deduplication

Dependencies:
- `%run ../config/settings`
- `%run ../auth/token_manager`
- `%run ../monitoring/request_hash`

In [ ]:
# %run ../config/settings
# %run ../auth/token_manager
# %run ../monitoring/request_hash

In [ ]:
import json as _json
import time
import urllib.error
import urllib.parse
import urllib.request

_MAX_RETRIES = 3


def spotify_api_call(
    method:   str,
    endpoint: str,
    params:   dict | None = None,
    body:     dict | None = None,
) -> tuple[dict, int, str]:
    """
    Make an authenticated request to the Spotify Web API.

    Args:
        method:   HTTP verb ("GET", "POST", …)
        endpoint: Path relative to SPOTIFY_API_BASE, e.g. "/me/player/recently-played"
        params:   Query-string parameters
        body:     JSON request body (for POST/PUT)

    Returns:
        (response_json, http_status, req_hash)
    """
    params = params or {}
    body   = body   or {}
    url    = f"{SPOTIFY_API_BASE}{endpoint}"
    req_hash = request_hash(method, url, params, body)

    return _call_with_retry(method, url, params, body, req_hash, attempt=0)


def _call_with_retry(
    method:   str,
    url:      str,
    params:   dict,
    body:     dict,
    req_hash: str,
    attempt:  int,
) -> tuple[dict, int, str]:
    full_url = f"{url}?{urllib.parse.urlencode(params)}" if params else url
    payload  = _json.dumps(body).encode() if body else None

    req = urllib.request.Request(
        full_url,
        data=payload,
        headers={
            "Authorization": f"Bearer {get_access_token()}",
            "Content-Type":  "application/json",
            "Accept":        "application/json",
        },
        method=method.upper(),
    )

    try:
        with urllib.request.urlopen(req) as resp:
            data = _json.loads(resp.read()) if resp.length != 0 else {}
            return data, resp.status, req_hash

    except urllib.error.HTTPError as e:
        if e.code == 429 and attempt < _MAX_RETRIES:
            wait = int(e.headers.get("Retry-After", 1))
            time.sleep(wait)
            return _call_with_retry(method, url, params, body, req_hash, attempt + 1)

        if e.code == 401 and attempt < 1:
            # Token was revoked or expired mid-session — invalidate and retry once
            global _auth_singleton
            if _auth_singleton is not None:
                _auth_singleton.invalidate()
            return _call_with_retry(method, url, params, body, req_hash, attempt + 1)

        error_body = e.read().decode(errors="replace")
        raise RuntimeError(
            f"Spotify API error {e.code} on {method.upper()} {url}: {error_body}"
        ) from e